<a href="https://colab.research.google.com/github/F1ameX/2025-ODS-NLP/blob/main/practice_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install and Import Libraries

In [54]:
!pip install --quiet catboost
!pip install --quiet gensim

In [55]:
import os
import re
import pandas as pd

from catboost import Pool, CatBoostClassifier

## Read Data

In [56]:
path = '/content/drive/MyDrive/ods-nlp_src/practice_2/'
train_data = pd.read_csv(os.path.join(path, 'train.csv'))
test_data = pd.read_csv(os.path.join(path, 'test.csv'))
print(f'Number of rows and columns in the train data set: {train_data.shape}')
print(f'Number of rows and columns in the test data set: {test_data.shape}')
train_data.head()

Number of rows and columns in the train data set: (48665, 2)
Number of rows and columns in the test data set: (12167, 2)


,rate,text
0,4,Очень понравилось. Были в начале марта с соба...
1,5,В целом магазин устраивает.\nАссортимент позво...
2,5,"Очень хорошо что открылась 5 ка, теперь не над..."
3,3,Пятёрочка громко объявила о том как она заботи...
4,3,"Тесно, вечная сутолока, между рядами трудно ра..."


In [57]:
train_data.groupby('rate').describe()

text                               
      count unique                top freq
rate                                      
1      4138   4130             Грязно    3
2      2410   2407  Отстойный магазин    2
3      6126   6070          Нормально    7
4      9922   9763               Норм   14
5     26069  24804    Хороший магазин  107

## Preparing the data and creating Catboost model

In [58]:
train_data['text'].sample(7)

,text
45535,Очень вежливые продавцы и санитарка. Ходишь ка...
38130,"Всё нормально. Хорошее обслуживание, вежливый ..."
14735,Самый лучший персонал)
11439,"Хороший, чистый магазин. Между рядами достат..."
35069,Все хорошо
41332,"Ассортимент маленький, но это болезнь почти вс..."
25858,"После обновления магазина, всё стало лучше. Ра..."


In [59]:
train_data['text'] = train_data['text'].str.lower()
train_data['text'] = train_data['text'].apply(lambda x: re.sub(r'[^\w\s]', '', x))

In [60]:
train_data['text'].sample(7)

,text
36413,иногда работает одна касса на весь магазинстал...
42568,магазин приличный
16313,хороший магазин и персонал вежливый и отзывчивый
32543,магазин чистый товар аккуратно выставлен на по...
7692,хороший магазин большой ассортимент 4 кассы к ...
10606,вежливый персонал обслуживание хорошее подскаж...
9733,если честно скудный выбор товаров еда есть в п...


In [64]:
X_train = train_data['text']
y_train = train_data['rate']

X_test = test_data['text']


model = CatBoostClassifier(
    iterations = 150,
    depth = 5,
    random_seed = 52
)

model.fit(
    X_train,
    y_train,
    text_features = [0],
    verbose=True
)

Learning rate set to 0.479263
0:	learn: 1.1007137	total: 5s	remaining: 12m 25s
1:	learn: 1.0168544	total: 8.53s	remaining: 10m 30s
2:	learn: 0.9655813	total: 12.2s	remaining: 9m 56s
3:	learn: 0.9423265	total: 16.9s	remaining: 10m 18s
4:	learn: 0.9326649	total: 20.8s	remaining: 10m 2s
5:	learn: 0.9233729	total: 24.3s	remaining: 9m 43s
6:	learn: 0.9179431	total: 27.8s	remaining: 9m 28s
7:	learn: 0.9128259	total: 33s	remaining: 9m 45s
8:	learn: 0.9086044	total: 36.5s	remaining: 9m 31s
9:	learn: 0.9025016	total: 40.1s	remaining: 9m 20s
10:	learn: 0.9006064	total: 44.4s	remaining: 9m 20s
11:	learn: 0.8992378	total: 48.8s	remaining: 9m 21s
12:	learn: 0.8950451	total: 52.4s	remaining: 9m 11s
13:	learn: 0.8932517	total: 55.9s	remaining: 9m 3s
14:	learn: 0.8898222	total: 1m 1s	remaining: 9m 9s
15:	learn: 0.8884537	total: 1m 4s	remaining: 9m
16:	learn: 0.8876079	total: 1m 8s	remaining: 8m 52s
17:	learn: 0.8866073	total: 1m 12s	remaining: 8m 48s
18:	learn: 0.8850125	total: 1m 16s	remaining: 8m 49

## Predict

In [65]:
# Preparing data in Pool format
dataset_test = Pool(
    data = X_test,
    text_features = [0]
)
predict_classes = model.predict(dataset_test)
predictions = predict_classes

## Create submission

In [66]:
sample_submission = pd.read_csv(os.path.join(path, 'sample_submission.csv'))
sample_submission['rate'] = predictions
sample_submission.head()

,index,rate
0,0,5
1,1,5
2,2,5
3,3,5
4,4,5


In [67]:
sample_submission.to_csv(os.path.join(path, 'submission.csv'), index=False)